# Step 4 — `rewritten_clean`: clean passages rewritten in LLM style

**Purpose** (protocol §6). PoisonedRAG's poisoned passages were written by an LLM; the clean passages are human-written. A detector could learn "sounds like an LLM = poisoned". This notebook rewrites each test clean passage with an LLM, keeping its facts and length, so the evaluation can test that shortcut: does the detector's false-alarm rate rise on clean passages that only *sound* generated?

**What it does**
1. Reads `data/to_rewrite.jsonl` (the 700 clean passages of the 140 test questions' n=0 groups, written by notebook 02).
2. Rewrites each with `Qwen/Qwen2.5-3B-Instruct` (the Step 1 generator), fp16, greedy decoding, one fixed prompt.
3. **Automatic checks** (fixed before running): non-empty, no preamble, every number in the original still present, length within 0.7–1.3× the original. Passages that fail are **excluded** and counted, never hand-fixed.
4. Writes `data/rewritten_clean.jsonl` (`passage_id`, `text`) for the passing rewrites, a report, and a **20-passage spot-check sheet** (seed 42) for a person to read.

**Then:** re-run `02_build_dataset` (adds the rewritten groups) and `03_distilbert` (scores only the new rows).

**Runtime.** Colab **T4 GPU**, about 30–45 minutes. Progress is saved after every batch, so a disconnect resumes.

## 1. Setup

In [ ]:
import os, sys, json, re, time, random, subprocess
try:
    gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
print(gpu or "NO GPU — switch the runtime to T4 GPU (Runtime -> Change runtime type)")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-U", "accelerate", "transformers>=4.45,<5"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/rag_trust"
else:
    DRIVE_ROOT = os.environ["RAG_TRUST_ROOT"]
DATA_DIR = f"{DRIVE_ROOT}/data"
SRC = f"{DATA_DIR}/to_rewrite.jsonl"
RAW = f"{DATA_DIR}/rewritten_clean_raw.jsonl"           # every generation, resumable
OUT = f"{DATA_DIR}/rewritten_clean.jsonl"                # passing rewrites only (read by notebook 02)
assert os.path.exists(SRC), "run notebooks/02_build_dataset.ipynb first"

CONFIG = {
    "generator": "Qwen/Qwen2.5-3B-Instruct", "dtype": "float16", "decoding": "greedy",
    "batch_size": 8, "max_new_tokens_factor": 1.6, "max_new_tokens_cap": 512,
    "length_ratio_range": [0.7, 1.3], "spot_check_n": 20, "seed": 42,
    "prompt": ("Rewrite the following passage in your own fluent, natural prose. "
               "Keep every fact, name, number and date exactly as it is. Do not add, remove or "
               "explain anything. Keep it about the same length (about {n_words} words). "
               "Reply with the rewritten passage only.\n\nPassage:\n{text}"),
}
json.dump(CONFIG, open(f"{DATA_DIR}/rewrite_config.json", "w"), indent=2)
ITEMS = [json.loads(l) for l in open(SRC)]
print(f"{len(ITEMS)} passages to rewrite")

## 2. Automatic checks (fixed before any rewrite is generated)

In [ ]:
NUM = re.compile(r"\d+(?:[.,]\d+)*")
PREAMBLE = re.compile(r"^(here is|here's|sure|certainly|rewritten passage|the rewritten)", re.I)

def n_words(s): return len(str(s).split())

def clean_output(s):
    s = s.strip().strip('"').strip()
    s = re.sub(r"^(rewritten passage|passage)\s*:\s*", "", s, flags=re.I)
    return " ".join(s.split())

def check(original, rewrite):
    """Returns a list of failed checks (empty = passes)."""
    fails = []
    if not rewrite:
        return ["empty"]
    if PREAMBLE.search(rewrite):
        fails.append("preamble")
    lo, hi = CONFIG["length_ratio_range"]
    ratio = n_words(rewrite) / max(n_words(original), 1)
    if not lo <= ratio <= hi:
        fails.append(f"length_ratio_{ratio:.2f}")
    norm = lambda x: x.replace(",", "")
    missing = {norm(x) for x in NUM.findall(original)} - {norm(x) for x in NUM.findall(rewrite)}
    if missing:
        fails.append("numbers_changed:" + ",".join(sorted(missing)[:5]))
    return fails

assert check("In 1999 it had 3,000 staff.", "It employed 3000 people in 1999.") == []
assert check("In 1999 it had 3,000 staff.", "It employed many people in 1999.")[0].startswith("numbers_changed")
assert check("one two three four five six seven eight nine ten", "one two")[0].startswith("length_ratio")
assert check("a b c d e f g h i j", "Here is the passage: a b c d e f") == ["preamble"]
print("checks: ok")

## 3. Rewrite (resumable)

In [ ]:
done = {}
if os.path.exists(RAW):
    for l in open(RAW):
        r = json.loads(l); done[r["passage_id"]] = r
todo = [x for x in ITEMS if x["passage_id"] not in done]
print(f"{len(done)} already rewritten, {len(todo)} to go")

if todo and os.environ.get("REWRITE_STUB") != "1":
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(CONFIG["generator"], padding_side="left")
    try:
        model = AutoModelForCausalLM.from_pretrained(CONFIG["generator"], dtype=torch.float16, device_map="auto")
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(CONFIG["generator"], torch_dtype=torch.float16, device_map="auto")
    model.eval()

    @torch.no_grad()
    def generate_batch(batch):
        texts = [tok.apply_chat_template([{"role": "user", "content": CONFIG["prompt"].format(
                     n_words=n_words(x["text"]), text=x["text"])}], add_generation_prompt=True, tokenize=False)
                 for x in batch]
        enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
        longest = max(len(tok(x["text"]).input_ids) for x in batch)
        max_new = min(int(CONFIG["max_new_tokens_factor"] * longest) + 20, CONFIG["max_new_tokens_cap"])
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None,
                             top_k=None, pad_token_id=tok.eos_token_id)
        return [tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True) for o in out]
else:
    def generate_batch(batch):            # offline dry run only: a trivial "rewrite"
        return ["In other words, " + x["text"] for x in batch]

t0 = time.time()
bs = CONFIG["batch_size"]
for i in range(0, len(todo), bs):
    batch = todo[i:i + bs]
    outs = generate_batch(batch)
    with open(RAW, "a") as f:
        for x, o in zip(batch, outs):
            rw = clean_output(o)
            rec = {"passage_id": x["passage_id"], "dataset": x["dataset"], "qid": x["qid"],
                   "original": x["text"], "rewrite": rw, "fails": check(x["text"], rw)}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n"); done[x["passage_id"]] = rec
    if (i // bs) % 10 == 0:
        print(f"  {min(i + bs, len(todo))}/{len(todo)} ({time.time() - t0:.0f}s)")
print(f"finished: {len(done)} rewrites")

## 4. Keep the passing rewrites, write the report and the spot-check sheet

In [ ]:
import collections
recs = [done[x["passage_id"]] for x in ITEMS]                    # fixed order
passing = [r for r in recs if not r["fails"]]
with open(OUT, "w") as f:
    for r in passing:
        f.write(json.dumps({"passage_id": r["passage_id"], "text": r["rewrite"]}, ensure_ascii=False) + "\n")

fail_kinds = collections.Counter(("length_ratio" if k.startswith("length_ratio") else k.split(":")[0]) for r in recs for k in r["fails"])
ratios = [n_words(r["rewrite"]) / max(n_words(r["original"]), 1) for r in passing]
REPORT = {"total": len(recs), "passing": len(passing), "excluded": len(recs) - len(passing),
          "excluded_by_check": dict(fail_kinds),
          "passing_by_dataset": dict(collections.Counter(r["dataset"] for r in passing)),
          "length_ratio_passing": {"median": round(float(sorted(ratios)[len(ratios) // 2]), 3) if ratios else None}}
json.dump(REPORT, open(f"{DATA_DIR}/rewrite_report.json", "w"), indent=2)
print(json.dumps(REPORT, indent=2))

# 20-passage spot-check sheet (seed 42) for a person to read: did any fact change?
import csv
rng = random.Random(CONFIG["seed"])
sample = rng.sample(passing, min(CONFIG["spot_check_n"], len(passing)))
with open(f"{DATA_DIR}/rewrite_spotcheck.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["passage_id", "dataset", "original", "rewrite", "verdict (ok / fact_changed / other)", "note"])
    for r in sample:
        w.writerow([r["passage_id"], r["dataset"], r["original"], r["rewrite"], "", ""])
print(f"spot-check sheet: {len(sample)} pairs -> data/rewrite_spotcheck.csv")

## 5. What to check, then next steps

1. **Report:** most of the 700 rewrites should pass. Note how many were excluded and why; they go in the report as a limitation, and are never fixed by hand.
2. **Spot-check (≈20 min, a person):** open `data/rewrite_spotcheck.csv` in Google Sheets (File → Import, untick "convert text to numbers"), read each pair, and fill in `verdict`: `ok`, `fact_changed` (any name, number, date or claim differs) or `other`. Report the count; if several facts changed, say so in the limitations.
3. **Re-run `02_build_dataset`** (CPU): it prints `rewritten passages: N` and adds the groups; `protocol checks: all passed`.
4. **Re-run `03_distilbert`** (T4): it scores only the new rows with the saved models; thresholds stay unchanged.
5. **Commit:** this notebook, and from Drive `data/`: `rewritten_clean.jsonl`, `rewrite_report.json`, `rewrite_config.json`, the filled-in `rewrite_spotcheck.csv`, and the rebuilt `passages.jsonl.gz` and `build_report.json`. Keep `rewritten_clean_raw.jsonl` on Drive (or commit it too: it records every exclusion).